# W12 · LLM 심화 통합 실습 — 근거, 구조화, 검증, 비교

오늘 노트북은 **하나의 파일 안에 여러 단계**로 구성됩니다.

1. 기본 예제: naive → grounded → structured → validation
2. 다중 LLM 비교: Gemini / ChatGPT / Claude
3. 추가 self-practice 예제 bank

> 목표: **그럴듯한 답**이 아니라 **근거가 있고, 다시 쓸 수 있고, 다시 확인된 답**을 만드는 것

## 오늘 핵심 개념 4개

### 1) Grounding = 근거 붙이기
- 내가 제공한 문서, 메모, 리뷰를 먼저 주고 답하게 하기

### 2) Structured output = 구조화 출력
- 자유문 대신 JSON/표/체크리스트로 결과 받기

### 3) Validation loop = 검증 루프
- 첫 답을 그대로 믿지 않고 다시 확인하기

### 4) Multi-LLM comparison = 모델 비교
- 같은 프롬프트라도 LLM마다 형식 준수, 과신, 근거 처리 방식이 다를 수 있음

---
## 섹션 1. 기본 실습 데이터 준비

지난주 상권분석 흐름을 이어서, **스터디카페 입지 후보를 판단하는 메모 8개**를 사용합니다.

In [ ]:
import json
import re
import pandas as pd

docs = pd.DataFrame([
    {"doc_id": "D1", "kind": "현장메모", "text": "A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다."},
    {"doc_id": "D2", "kind": "임대료메모", "text": "A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다."},
    {"doc_id": "D3", "kind": "현장메모", "text": "B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다."},
    {"doc_id": "D4", "kind": "경쟁메모", "text": "B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다."},
    {"doc_id": "D5", "kind": "현장메모", "text": "C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다."},
    {"doc_id": "D6", "kind": "학생설문", "text": "학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다."},
    {"doc_id": "D7", "kind": "운영규정", "text": "밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다."},
    {"doc_id": "D8", "kind": "인터뷰", "text": "시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다."},
])

docs

,doc_id,kind,text
0,D1,현장메모,A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이...
1,D2,임대료메모,A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이...
2,D3,현장메모,B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 ...
3,D4,경쟁메모,"B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카..."
4,D5,현장메모,C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이...
5,D6,학생설문,"학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 ..."
6,D7,운영규정,밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지...
7,D8,인터뷰,시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳...


In [ ]:
question = "학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘."
print(question)

학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘.


In [ ]:
context_text = "\n".join([f"[{row.doc_id}] ({row.kind}) {row.text}" for _, row in docs.iterrows()])
print(context_text)

[D1] (현장메모) A구역은 점심 유동인구가 많고 도서관과 3분 거리다. 학생이 1~2시간 머물 공간이 부족하다는 말이 있었다.
[D2] (임대료메모) A구역 월세는 210만원 수준으로 높다. 반경 200m 안에 테이크아웃 카페 4곳이 이미 있다.
[D3] (현장메모) B구역은 월세가 130만원 수준으로 저렴하고 비교적 조용하다. 다만 밤 8시 이후 유동인구가 급격히 줄어든다.
[D4] (경쟁메모) B구역 주변에는 프랜차이즈 카페 1곳만 있고, 오래 머물 수 있는 스터디 좌석형 카페는 확인되지 않았다.
[D5] (현장메모) C구역은 지하철역 출구 앞이라 유동인구가 매우 많다. 대신 차량 소음과 보행 소음이 크고 회전이 빠르다.
[D6] (학생설문) 학생들은 조용한 분위기, 콘센트, 2시간 이상 머물 수 있는 좌석을 가장 중요하게 꼽았다.
[D7] (운영규정) 밤 10시 이후 외부 소음 유발 활동은 제한된다. 늦은 시간 행사형 운영은 적합하지 않다.
[D8] (인터뷰) 시험기간에는 도서관 주변 좌석 부족 민원이 반복된다. 특히 오후 3시 이후 앉을 곳이 없다는 의견이 많았다.


---
## 섹션 2. naive 답변 vs grounded 답변

같은 질문을 두 번 던져 차이를 봅니다.

In [ ]:
naive_prompt = f'''
너는 친절한 분석 도우미야.

질문:
{question}
'''

print(naive_prompt)


너는 친절한 분석 도우미야.

질문:
학교 앞에 20석 규모 스터디카페를 열 때 가장 적합한 후보 구역 1곳을 고르고 이유와 위험을 설명해줘.



In [ ]:
naive_answer = '''
여기에 naive 답변을 붙여넣으세요.
'''

print(naive_answer[:400])

In [ ]:
grounded_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
답변에는 근거가 되는 문서 ID를 반드시 괄호로 붙여줘.
문서에 없는 내용은 추측하지 말고, 모르면 모른다고 말해줘.

문서 묶음:
{context_text}

질문:
{question}
'''

print(grounded_prompt)

In [ ]:
grounded_answer = '''
여기에 grounded 답변을 붙여넣으세요.
'''

print(grounded_answer[:500])

#### 비교 메모
- naive 답변의 한계:
- grounded 답변의 장점:
- grounded 답변에도 아직 부족한 점:

---
## 섹션 3. 구조화 출력 — JSON으로 받기

In [ ]:
json_schema = {
    "candidate_zone": "A구역/B구역/C구역 중 하나",
    "fit_reason": "한 줄 설명",
    "evidence_ids": ["D1", "D6"],
    "risk": "위험 한 줄",
    "confidence_1_to_5": 3,
    "missing_info": "추가로 필요하지만 현재 없는 정보"
}

print(json.dumps(json_schema, ensure_ascii=False, indent=2))

NameError: name 'json' is not defined

In [ ]:
structured_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
문서에 없는 내용은 추측하지 말고 missing_info에 적어줘.
반드시 JSON만 출력해줘. 설명 문장은 쓰지 마.

문서 묶음:
{context_text}

질문:
{question}

JSON 형식:
{json.dumps(json_schema, ensure_ascii=False, indent=2)}
'''

print(structured_prompt)

In [ ]:
json_text = r'''
{
  "candidate_zone": "",
  "fit_reason": "",
  "evidence_ids": [],
  "risk": "",
  "confidence_1_to_5": 0,
  "missing_info": ""
}
'''

print(json_text)

In [ ]:
parsed = json.loads(json_text)
pd.DataFrame([parsed])

#### 구조화 출력 메모
- 자유문보다 좋은 점:
- 여전히 조심해야 할 점:

---
## 섹션 4. 검증 루프

In [ ]:
allowed_doc_ids = set(docs["doc_id"])
allowed_zones = {"A구역", "B구역", "C구역"}

def validate_answer(obj):
    problems = []
    required_keys = {
        "candidate_zone",
        "fit_reason",
        "evidence_ids",
        "risk",
        "confidence_1_to_5",
        "missing_info",
    }

    missing = required_keys - set(obj.keys())
    if missing:
        problems.append(f"키 누락: {sorted(missing)}")

    zone = obj.get("candidate_zone")
    if zone not in allowed_zones:
        problems.append(f"candidate_zone 값 이상: {zone}")

    evidence_ids = obj.get("evidence_ids", [])
    if not isinstance(evidence_ids, list):
        problems.append("evidence_ids는 리스트여야 함")
    else:
        bad_ids = [x for x in evidence_ids if x not in allowed_doc_ids]
        if bad_ids:
            problems.append(f"존재하지 않는 evidence_ids: {bad_ids}")

    confidence = obj.get("confidence_1_to_5")
    if not isinstance(confidence, int) or not (1 <= confidence <= 5):
        problems.append(f"confidence_1_to_5 값 이상: {confidence}")

    if len(str(obj.get("fit_reason", "")).strip()) < 10:
        problems.append("fit_reason이 너무 짧음")

    if len(str(obj.get("risk", "")).strip()) < 5:
        problems.append("risk가 너무 짧음")

    return problems

problems = validate_answer(parsed)
if problems:
    print("형식/검증 이슈 발견")
    for p in problems:
        print("-", p)
else:
    print("기본 형식 검증 통과")

In [ ]:
review_prompt = f'''
너는 엄격한 검토자야.
아래 문서 묶음과 JSON 답변을 보고,
1) 근거가 약한 부분 2개
2) 과신한 부분 1개
3) 추가로 필요한 정보 2개
를 짧게 적어줘.

문서 묶음:
{context_text}

JSON 답변:
{json_text}
'''

print(review_prompt)

#### 검토 결과
- 근거가 약한 부분:
- 과신한 부분:
- 추가로 필요한 정보:

In [ ]:
revised_json_text = r'''
{
  "candidate_zone": "",
  "fit_reason": "",
  "evidence_ids": [],
  "risk": "",
  "confidence_1_to_5": 0,
  "missing_info": ""
}
'''

revised = json.loads(revised_json_text)
pd.DataFrame([revised])

In [ ]:
revised_problems = validate_answer(revised)
if revised_problems:
    print("수정본에도 이슈가 남아 있음")
    for p in revised_problems:
        print("-", p)
else:
    print("수정본 형식 검증 통과")

---
## 섹션 5. 다중 LLM 비교 실험

같은 프롬프트를 **Gemini / ChatGPT / Claude**에 던져 결과 차이를 봅니다.

> 3개가 어렵다면 2개만 비교해도 됩니다.

In [ ]:
model_answers = {
    "Gemini": '''여기에 Gemini grounded 답변을 붙여넣으세요.''',
    "ChatGPT": '''여기에 ChatGPT grounded 답변을 붙여넣으세요.''',
    "Claude": '''여기에 Claude grounded 답변을 붙여넣으세요.''',
}

comparison_rows = []
for model_name, text in model_answers.items():
    evidence_ids = sorted(set(re.findall(r"D\d+", text)))
    comparison_rows.append({
        "model": model_name,
        "char_len": len(text.strip()),
        "evidence_count": len(evidence_ids),
        "evidence_ids": ", ".join(evidence_ids),
    })

pd.DataFrame(comparison_rows)

#### grounded 답변 모델 비교 메모
- 어떤 모델이 근거를 가장 잘 붙였나:
- 어떤 모델이 가장 장황했나:
- 어떤 모델이 가장 과신했나:

In [ ]:
model_json_answers = {
    "Gemini": r'''{"candidate_zone":"","fit_reason":"","evidence_ids":[],"risk":"","confidence_1_to_5":0,"missing_info":""}''',
    "ChatGPT": r'''{"candidate_zone":"","fit_reason":"","evidence_ids":[],"risk":"","confidence_1_to_5":0,"missing_info":""}''',
    "Claude": r'''{"candidate_zone":"","fit_reason":"","evidence_ids":[],"risk":"","confidence_1_to_5":0,"missing_info":""}''',
}

def try_parse_json(text):
    try:
        obj = json.loads(text)
        return True, obj
    except Exception as e:
        return False, str(e)

rows = []
for model_name, text in model_json_answers.items():
    ok, result = try_parse_json(text)
    row = {"model": model_name, "json_parse_ok": ok}
    if ok:
        row["keys"] = ", ".join(sorted(result.keys()))
        row["validation_issues"] = " / ".join(validate_answer(result)) or "통과"
    else:
        row["keys"] = "-"
        row["validation_issues"] = str(result)
    rows.append(row)

pd.DataFrame(rows)

#### structured 출력 모델 비교 메모
- 어떤 모델이 JSON 규칙을 가장 잘 지켰나:
- 어떤 모델이 missing_info를 가장 솔직하게 썼나:
- 어떤 모델 결과를 실무에 다시 쓰기 가장 쉬웠나:

---
## 섹션 6. 추가 self-practice 예제 bank

시간이 남으면 아래 예제 중 1개를 골라 같은 흐름으로 다시 해 보세요.

In [ ]:
challenge_cases = {
    "review_summary": {
        "title": "카페 리뷰 요약기",
        "question": "리뷰 묶음을 보고 장점 3개, 불만 3개, 가장 먼저 고칠 점 1개를 정리해줘.",
        "schema": {
            "top_strengths": [],
            "top_issues": [],
            "quick_fix": "",
            "evidence_ids": [],
            "confidence_1_to_5": 0,
        },
        "docs": [
            {"doc_id": "R1", "kind": "리뷰", "text": "좌석이 편하고 콘센트가 많아서 좋았다."},
            {"doc_id": "R2", "kind": "리뷰", "text": "음악 소리가 커서 오래 공부하기 어렵다."},
            {"doc_id": "R3", "kind": "리뷰", "text": "직원이 친절하고 음료가 빨리 나왔다."},
            {"doc_id": "R4", "kind": "리뷰", "text": "화장실 청결이 아쉬웠다."},
            {"doc_id": "R5", "kind": "리뷰", "text": "시험기간에 자리가 부족했다."},
            {"doc_id": "R6", "kind": "리뷰", "text": "디저트는 평범하지만 공간은 조용했다."},
        ],
    },
    "survey_sort": {
        "title": "자유응답 설문 분류기",
        "question": "설문 응답을 칭찬/불만/개선요청으로 분류하고, 가장 우선순위 높은 개선요청 2개를 골라줘.",
        "schema": {
            "top_priority_requests": [],
            "categories": [],
            "evidence_ids": [],
            "confidence_1_to_5": 0,
        },
        "docs": [
            {"doc_id": "S1", "kind": "설문", "text": "실습은 재미있었지만 속도가 조금 빨랐다."},
            {"doc_id": "S2", "kind": "설문", "text": "예제가 많아서 좋았다."},
            {"doc_id": "S3", "kind": "설문", "text": "과제가 헷갈려서 제출 예시가 더 필요하다."},
            {"doc_id": "S4", "kind": "설문", "text": "교수님 설명이 차분해서 좋았다."},
            {"doc_id": "S5", "kind": "설문", "text": "용어 설명을 더 쉽게 해주면 좋겠다."},
            {"doc_id": "S6", "kind": "설문", "text": "실습 시간이 더 길었으면 좋겠다."},
        ],
    },
    "team_roles": {
        "title": "팀플 역할 배치 도우미",
        "question": "팀원 4명에게 발표/자료조사/디자인/일정관리 역할을 배정하고, 충돌 위험이 있으면 함께 적어줘.",
        "schema": {
            "assignments": [],
            "risk": "",
            "evidence_ids": [],
            "confidence_1_to_5": 0,
        },
        "docs": [
            {"doc_id": "T1", "kind": "자기소개", "text": "민수는 발표 경험이 많고 사람 앞에서 말하는 걸 덜 두려워한다."},
            {"doc_id": "T2", "kind": "자기소개", "text": "지연은 꼼꼼하고 일정표 정리를 잘한다."},
            {"doc_id": "T3", "kind": "자기소개", "text": "하준은 포토샵과 PPT 디자인 경험이 있다."},
            {"doc_id": "T4", "kind": "자기소개", "text": "서윤은 자료 찾는 속도가 빠르고 문서 정리를 잘한다."},
        ],
    },
    "rule_checklist": {
        "title": "규정 체크리스트 만들기",
        "question": "규정을 해야 할 것/하면 안 되는 것/확인 필요한 것으로 나누고, 신입생이 실수하기 쉬운 항목 3개를 골라줘.",
        "schema": {
            "must_do": [],
            "must_not_do": [],
            "needs_confirmation": [],
            "risk_points": [],
            "evidence_ids": [],
        },
        "docs": [
            {"doc_id": "P1", "kind": "규정", "text": "동아리실 사용 후 전등과 냉난방기를 반드시 끈다."},
            {"doc_id": "P2", "kind": "규정", "text": "외부인 출입은 사전 승인 없이 허용되지 않는다."},
            {"doc_id": "P3", "kind": "규정", "text": "음식물 반입은 가능하나 냄새가 강한 음식은 자제한다."},
            {"doc_id": "P4", "kind": "규정", "text": "기자재 대여는 담당자 확인 후 기록지 작성이 필요하다."},
        ],
    },
}

pd.DataFrame([
    {"case_key": k, "title": v["title"], "doc_count": len(v["docs"]), "question": v["question"]}
    for k, v in challenge_cases.items()
])

In [ ]:
selected_case = "review_summary"   # 다른 예제를 하려면 여기 값을 바꾸세요.

case = challenge_cases[selected_case]
case_docs = pd.DataFrame(case["docs"])
case_question = case["question"]
case_schema = case["schema"]
case_context_text = "\n".join([f"[{row.doc_id}] ({row.kind}) {row.text}" for _, row in case_docs.iterrows()])

print("선택한 예제:", case["title"])
print(case_question)
case_docs

In [ ]:
case_grounded_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
답변에는 근거 ID를 붙여줘.
문서에 없는 내용은 추측하지 말고 모르면 모른다고 말해줘.

문서 묶음:
{case_context_text}

질문:
{case_question}
'''

print(case_grounded_prompt)

In [ ]:
case_structured_prompt = f'''
너는 친절한 분석 도우미야.
아래 문서 묶음만 근거로 사용해서 답해줘.
반드시 JSON만 출력해줘.

문서 묶음:
{case_context_text}

질문:
{case_question}

JSON 형식:
{json.dumps(case_schema, ensure_ascii=False, indent=2)}
'''

print(case_structured_prompt)

In [ ]:
case_model_answers = {
    "Gemini": '''여기에 답변을 붙여넣으세요.''',
    "ChatGPT": '''여기에 답변을 붙여넣으세요.''',
}

pd.DataFrame([
    {
        "model": name,
        "char_len": len(text.strip()),
        "evidence_ids": ", ".join(sorted(set(re.findall(r"[A-Z]\d+", text)))),
    }
    for name, text in case_model_answers.items()
])

#### 추가 예제 실습 메모
- 내가 고른 예제:
- 어떤 LLM 2개를 비교했는가:
- 차이가 가장 컸던 부분:
- 내가 더 믿을 수 있다고 판단한 결과와 이유:

---
## 섹션 7. 질문 유형 바꿔 보기

같은 문서 묶음이라도 질문을 바꾸면 결과가 달라집니다.

In [ ]:
prompt_bank = [
    "가장 적합한 후보 1개를 고르고 이유를 설명해줘.",
    "후보를 순위대로 1위~3위까지 정리해줘.",
    "장점 / 위험 / 추가 필요 정보로 표를 만들어줘.",
    "가장 과신하면 안 되는 부분 2개를 짚어줘.",
    "의사결정 전에 꼭 더 확인해야 할 정보 3개를 적어줘.",
    "초보자도 바로 이해할 수 있는 체크리스트로 바꿔줘.",
]

pd.DataFrame({"extra_prompts": prompt_bank})

#### 질문 변형 메모
- 어떤 질문이 가장 좋은 답을 만들었는가:
- 어떤 질문은 너무 넓어서 오히려 답이 흐려졌는가:

---
## 섹션 8. 오늘의 정리

### 오늘 배운 것 3줄 정리
1.
2.
3.

### AI 사용 내역 1줄
- 사용 도구:
- 핵심 프롬프트:
- 내가 직접 검증한 것:

### 최종 비교 메모
- 가장 믿을 수 있었던 모델:
- 가장 다시 쓰기 편했던 출력 형식:
- 다음에 내 프로젝트에 가져갈 기술 1개:

### Git Sync: Push Notebook to GitHub
This section handles the authentication and synchronization with your GitHub repository.

In [3]:
import os
from google.colab import userdata

# Configuration
REPO_URL = "https://github.com/qefdva/programing2026"
REPO_NAME = "programing2026"
USER_NAME = "qefdva"
USER_EMAIL = "qefdva@users.noreply.github.com"

# 1. 여기에 복사한 토큰을 붙여넣으세요 (예: "ghp_xxxx...")
GITHUB_TOKEN = "HIDDEN_TOKEN"

try:
    if GITHUB_TOKEN == "YOUR_TOKEN_HERE" or not GITHUB_TOKEN:
        # 만약 Colab Secrets(🔑)에 저장했다면 아래 코드가 작동합니다
        try:
            GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
        except:
            print("토큰이 입력되지 않았습니다. 'YOUR_TOKEN_HERE' 자리에 토큰을 붙여넣거나 Secrets에 등록하세요.")

    if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_TOKEN_HERE":
        AUTH_REPO_URL = REPO_URL.replace("https://", f"https://{USER_NAME}:{GITHUB_TOKEN}@")

        if not os.path.exists(REPO_NAME):
            !git clone {AUTH_REPO_URL}
        else:
            %cd {REPO_NAME}
            !git pull
            %cd ..

        print("✅ 저장소 연결 성공!")
except Exception as e:
    print(f"오류 발생: {e}")

Cloning into 'programing2026'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
✅ 저장소 연결 성공!


In [6]:
import os
import json
from google.colab import _message

# 1. 현재 노트북의 모든 셀 데이터를 가져와서 파일로 저장
def save_current_notebook(filename):
    try:
        # Colab 내부 메시징을 통해 노트북 구조 가져오기
        resp = _message.blocking_request('get_ipynb', request={}, timeout_sec=30)
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(resp['ipynb'], f, ensure_ascii=False, indent=2)
        return True
    except Exception as e:
        print(f"노트북 저장 중 오류 발생: {e}")
        return False

notebook_filename = "programing2026.ipynb"

if os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

    # 2. 현재 상태 저장
    if save_current_notebook(notebook_filename):
        print(f"✅ {notebook_filename} 저장 완료")

        # 3. Git 설정 및 푸시
        !git config --global user.email "{USER_EMAIL}"
        !git config --global user.name "{USER_NAME}"
        !git add {notebook_filename}
        !git commit -m "Final update from Colab (Auto-saved)"
        !git push origin main
        print(f"\n🚀 깃허브 푸시 성공! {REPO_URL} 에서 확인하세요.")
    else:
        print("❌ 노트북 파일 생성 실패")

    %cd ..
else:
    print("저장소 폴더를 찾을 수 없습니다. 설정 셀을 먼저 실행해 주세요.")

/content/programing2026
업로드된 노트북 파일을 찾을 수 없습니다. 파일을 먼저 Colab에 업로드해주세요.

❌ 푸시할 노트북 파일이 저장소 폴더에 없습니다.
/content
